In [6]:
import sys
import os
import yaml
import warnings

warnings.filterwarnings("ignore") 

# 1. 경로 설정 및 모듈 임포트
PROJECT_ROOT = "/home/taehoon/sprint-ai-mid-project_team3"
if os.path.abspath(PROJECT_ROOT) not in sys.path:
    sys.path.append(os.path.abspath(PROJECT_ROOT))

from src.retriever_factory import create_retriever

# 설정 파일 로드
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config/default.yaml")
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"설정 파일이 존재하지 않습니다: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config = yaml.safe_load(f)

retrieval_config = full_config.get("retrieval") or full_config.get("retriever", {})

# 2. 지정된 경로에서 sample_rfp.txt 파일을 읽어와 청킹 수행
FILE_PATH = os.path.join(PROJECT_ROOT, "samples/raw/sample_rfp.txt")
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"지정한 경로에 파일이 존재하지 않습니다: {FILE_PATH}")

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

chunks_for_factory = []
for idx, paragraph in enumerate(paragraphs):
    agency = "과학기술정보통신부" if idx % 2 == 0 else "산업통상자원부"
    project = "지능형_AI_플랫폼_구축" if idx % 2 == 0 else "에너지_빅데이터_분석"
    
    chunks_for_factory.append({
        "text": paragraph, 
        "agency": agency, 
        "project_name": project,
        "doc_id": f"DOC_RFP_{idx:03d}"
    })
        
print(f"총 {len(chunks_for_factory)}개의 텍스트 문서 청크가 준비되었습니다.")

# 3. 통합 리트리버 생성 및 데이터 적재
default_config = retrieval_config.copy()
if "profiles" in default_config and "local" in default_config["profiles"]:
    default_config["profiles"]["local"]["collection_name"] = "total_rfp_collection"

retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=default_config, 
    profile="local"
)
print("Chroma DB에 sample_rfp 데이터 적재 프로세스 완료!\n")


# 4. 규격 변경 반영 정밀 검증
query_str = "사업의 추진 배경 및 기대 효과를 알려주세요"

print("=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===")
results = retriever.search(query=query_str, top_k=3)
for d in results:
    doc_id = d.metadata.get('doc_id', 'N/A')
    agency_name = d.metadata.get('agency', 'N/A')
    print(f"[{doc_id} / {agency_name}](Score: {d.score:.4f}) {d.text[:60]}...")

print("\n=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===")
results_filtered = retriever.search(query=query_str, top_k=2, agency="과학기술정보통신부")
for d in results_filtered:
    agency_name = d.metadata.get('agency', 'N/A')
    project_name = d.metadata.get('project_name', 'N/A')
    print(f"[{agency_name} / {project_name}](Score: {d.score:.4f}) {d.text[:60]}...")

print("\n=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===")
results_doc = retriever.search(query=query_str, top_k=1, doc_id="DOC_RFP_001")
if results_doc:
    # [수정 전송 완료] 리스트 타깃에서 정확히 첫 요소인 [0]을 추출하도록 고정했습니다.
    target_doc = results_doc[0] if isinstance(results_doc, list) else results_doc
    doc_id = target_doc.metadata.get('doc_id', 'N/A')
    print(f"[{doc_id}](Score: {target_doc.score:.4f}) {target_doc.text[:100]}...")


print("\n" + "="*50)
print("=== 5. Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===")
print("="*50)

# Multi-Collection 전용 데이터 분할
msit_chunks = [c for c in chunks_for_factory if c["agency"] == "과학기술정보통신부"]
motie_chunks = [c for c in chunks_for_factory if c["agency"] == "산업통상자원부"]

msit_run_config = retrieval_config.copy()
if "profiles" in msit_run_config and "local" in msit_run_config["profiles"]:
    msit_run_config["profiles"]["local"]["collection_name"] = "msit_exclusive_collection"

motie_run_config = retrieval_config.copy()
if "profiles" in motie_run_config and "local" in motie_run_config["profiles"]:
    motie_run_config["profiles"]["local"]["collection_name"] = "motie_exclusive_collection"

# 1. 과기부 전용 리트리버 독립 생성 및 적재
msit_retriever = create_retriever(chunks=msit_chunks, retrieval_config=msit_run_config, profile="local")
# 2. 산자부 전용 리트리버 독립 생성 및 적재
motie_retriever = create_retriever(chunks=motie_chunks, retrieval_config=motie_run_config, profile="local")

hybrid_query = "사업 추진 배경" 
print(f"-> 과기부 컬렉션 대상 순수 검색 질의: '{hybrid_query}'")

# 하이브리드 독립 컬렉션 쿼리 직접 테스트
results_hybrid = msit_retriever.search(query=hybrid_query, top_k=2)

print(f"Multi-Collection 하이브리드 테스트 호출 완료 (반환된 결과: {len(results_hybrid)}건)")
for idx, d in enumerate(results_hybrid):
    p_name = d.metadata.get('project_name', 'N/A')
    print(f"  [{idx+1}] {p_name} -> {d.text[:50]}...")

총 16개의 텍스트 문서 청크가 준비되었습니다.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chroma DB에 sample_rfp 데이터 적재 프로세스 완료!

=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===
[DOC_RFP_002 / N/A](Score: 0.7616) 1. 사업 개요...
[DOC_RFP_002 / N/A](Score: 0.7616) 1. 사업 개요...
[DOC_RFP_002 / N/A](Score: 0.7616) 1. 사업 개요...

=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===

=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===
[DOC_RFP_001](Score: 1.2802) 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다....

=== 5. Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

-> 과기부 컬렉션 대상 순수 검색 질의: '사업 추진 배경'
Multi-Collection 하이브리드 테스트 호출 완료 (반환된 결과: 2건)
  [1] 지능형_AI_플랫폼_구축 -> 1. 사업 개요...
  [2] 지능형_AI_플랫폼_구축 -> 1. 사업 개요...
